## Experiment No: 3
## Experiment Title: Edit Distance and Spelling Correction

**Name:** Himanshu Jadhav  
**Roll Number:** TE-33

### Step 1: Import Libraries

In [1]:
import nltk
import pandas as pd

nltk.download('words', quiet=True)

from nltk.corpus import words as nltk_wprds

### Step 2: Build a Small Vocabulary with Word Frequencies

In [2]:
vocabulary = {
    "hello"    : 500, "world"      : 300, "python"   : 250, "language"    : 200,
    "natural"  : 150, "processing" : 180, "learning" : 220, "machine"     : 210,
    "computer" : 190, "science"    : 170, "helpful"  : 90 , "hello world" : 5  ,
    "spelling" : 60 , "correction" : 55 , "college"  : 120, "student"     : 140
}

print("Vocabulary size:", len(vocabulary))
print(list(vocabulary.items())[:5])

Vocabulary size: 16
[('hello', 500), ('world', 300), ('python', 250), ('language', 200), ('natural', 150)]


### Step 3: Implement Edit Distance (Dynamic Programming)

In [5]:
def edit_distance(str1, str2):
    m, n = len(str1), len(str2)

    dp   = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for j in range(1, m + 1):
        for j in range(1, n + 1):
            if str1[i - 1] == str2[j -1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],
                    dp[i][j - 1],
                    dp[i - 1][j - 1]
                )
    return dp[m][n], dp

distance, matrix = edit_distance("kitten", "sitting")

print(f"Edit distance between 'kitten' and 'sitting': {distance}" )

print("\nDistance Matrix: ")
print(pd.DataFrame(matrix))

Edit distance between 'kitten' and 'sitting': 1

Distance Matrix: 
   0  1  2  3  4  5  6  7
0  0  1  2  3  4  5  6  7
1  1  0  0  0  0  0  0  0
2  2  0  0  0  0  0  0  0
3  3  0  0  0  0  0  0  0
4  4  0  0  0  0  0  0  0
5  5  0  0  0  0  0  0  0
6  6  1  1  1  1  1  0  1


### Step 4: Take a Misspelled Word

In [6]:
misspelled_word = 'helo'
print("Misspelled Word: ", misspelled_word)

Misspelled Word:  helo


### Step 5: Generate Candidate Corrections

In [8]:
def generate_candidates(word, vocab):
    candidates = []
    for vocab_word in vocab:
        dist, _ = edit_distance(word, vocab_word)
        candidates.append((vocab_word, dist))
    return candidates

candidates    = generate_candidates(misspelled_word, vocabulary)
candidates_df = pd.DataFrame(candidates, columns=["Candidate Word", "Edit Distance"])
candidates_df = candidates_df.sort_values("Edit Distance").reset_index(drop=True)

### Step 6: Rank Candidates using Distance and Frequency

In [9]:
candidates_df["Frequency"] = candidates_df["Candidate Word"].map(vocabulary)

# Rank primarily by edit distance (lower is better), then by frequency (higher is better)
ranked_df = candidates_df.sort_values(
    by=["Edit Distance", "Frequency"], ascending=[True, False]
).reset_index(drop=True)

print(ranked_df.head(5))

best_correction = ranked_df.iloc[0]["Candidate Word"]
print("\nMost probable correction for '{}': '{}'".format(misspelled_word, best_correction))

  Candidate Word  Edit Distance  Frequency
0          hello              0        500
1          world              1        300
2         python              1        250
3       learning              1        220
4        machine              1        210

Most probable correction for 'helo': 'hello'


### Step 7: Test on Multiple Misspelled Words

In [10]:
test_words = ["helo", "wrold", "pythom", "sciense", "leanring"]

results = []
for w in test_words:
    cands = generate_candidates(w, vocabulary)
    cand_df = pd.DataFrame(cands, columns=["word", "dist"])
    cand_df["freq"] = cand_df["word"].map(vocabulary)
    cand_df = cand_df.sort_values(by=["dist", "freq"], ascending=[True, False])
    best = cand_df.iloc[0]["word"]
    results.append([w, best, cand_df.iloc[0]["dist"]])

results_df = pd.DataFrame(results, columns=["Misspelled Word", "Suggested Correction", "Edit Distance"])
print(results_df)

  Misspelled Word Suggested Correction  Edit Distance
0            helo                hello              0
1           wrold                world              0
2          pythom                hello              1
3         sciense              machine              0
4        leanring             learning              0


### Final Output

In [11]:
print("Experiment Completed Successfully\n")
print(results_df)

Experiment Completed Successfully

  Misspelled Word Suggested Correction  Edit Distance
0            helo                hello              0
1           wrold                world              0
2          pythom                hello              1
3         sciense              machine              0
4        leanring             learning              0
